# Sv1: locked evidence walkthrough

This notebook reads final artifacts; it does not train, select, or rescore confirmation.

In [1]:
from pathlib import Path
import json, pandas as pd
ROOT = Path.cwd()
decision = json.loads((ROOT/'CONFIRMATION_DECISION.json').read_text())
decision

{'baseline': 'patchtst',
 'bootstrap_summary_sha256': '41674e231b18360ab9735ac3db8f62cc1e76abfac007eabd8ee141d5bb7a55a1',
 'completed_at': '2026-07-20T20:54:13.769556+00:00',
 'conditions': {'at_least_six_dataset_wins': False,
  'ci_excludes_zero_positive': False,
  'integrity_lock_verified': True,
  'mae_and_mase_not_significantly_harmed': True,
  'point_exceeds_sesoi': True},
 'confirmation_metrics_sha256': '608cade781fd046c9dcd1ad04c58c6a31deddfbd7e9fe99520bd2ba0bc6a916b',
 'dataset_wins': 5,
 'decision': 'inconclusive',
 'decision_hash': '94dcedbd8f6d48666aa18f1e83e2c1d3909ce3bc642df55dc4a1924db5a14d78',
 'lock_hash': '684cf557800bb060f0a40f6647ef2cf98ae17fec12ba629aa7aad84027ac5907',
 'primary_estimate': 0.012479276496508606,
 'primary_lower_95': -0.0030810659059164703,
 'primary_method': 'sv1_xfit_router',
 'primary_upper_95': 0.04082886716430144,
 'protocol_id': 'sv1-v5-retrieval-audit-20260721',
 'sesoi': 0.01}

## Dataset-level confirmatory effects

In [2]:
metrics = pd.read_csv(ROOT/'results/tables/confirmation_dataset_metrics.csv')
metrics[metrics.method.isin(['sv1_xfit_router','patchtst'])].pivot(index='dataset', columns='method', values='mse').assign(relative_reduction=lambda x: 1-x.sv1_xfit_router/x.patchtst)

method,patchtst,sv1_xfit_router,relative_reduction
dataset,,,
bitcoin,9390.324781,9376.004450,0.001525
fred_md,40.255215,39.707756,0.013600
kdd_air,1.105738,1.101919,0.003454
m4_hourly,0.500942,0.500942,0.000000
m4_weekly,2.600377,2.445608,0.059518
nn5,0.830744,0.832233,-0.001792
pedestrian,0.269494,0.270555,-0.003936
rideshare,2.723477,2.723477,0.000000
temperature_rain,4.469382,4.549051,-0.017826


## Bootstrap decision and metric robustness

In [3]:
bootstrap = pd.read_csv(ROOT/'results/tables/confirmation_bootstrap_summary.csv')
bootstrap.query("method == 'sv1_xfit_router'")

,method,metric,estimate,lower_95,upper_95,probability_positive,replicates
11,sv1_xfit_router,mse,0.012479,-0.003081,0.040829,0.93450,4000
13,sv1_xfit_router,mae,0.010644,0.000721,0.024458,0.98375,4000
14,sv1_xfit_router,mase,0.002189,-0.026165,0.023487,0.61675,4000


## Leakage audit

In [4]:
leakage = pd.read_csv(ROOT/'results/tables/leakage_audit.csv')
leakage.groupby(['bank','mode']).mse.median().unstack()

mode,base,exact_self_inclusion,leave_one_out
bank,,,
crossfit,0.611008,2.893863e-16,0.307927
insample,0.573443,2.109529e-16,0.284289


## Registered figure inventory

In [5]:
figures = pd.read_csv(ROOT/'results/tables/figure_manifest.csv')
assert len(figures) == 30
figures[['figure','slug','png_sha256','pdf_sha256']]

,figure,slug,png_sha256,pdf_sha256
0,1,study_design,b2bbbc248ec95f7239e0d6064ceeba9126f930731e871d...,9f529bed99090423cb9d57ca0d7171886b089ef845a101...
1,2,temporal_roles,66e1ecde2e3d38d79c27cf7cf1d84252e2845f96f18720...,bc80f9bbc2d4c38fe3cea5da850fe158e9aedc9a42c888...
2,3,dataset_atlas,cc4b121a7ce10bb9655a1fad86d075ccc54ec7ca30ecaa...,587c5d23fc2fe83364807bace0e41673c8c1f1767cde6b...
3,4,window_inventory,31178ea33015093b5d2a16af0415e2b716caee5ea9f7ca...,99851b3e2db1d8f117a9a7f6ef9b9e33514fa8991c0b67...
4,5,series_lengths,38e5a1f95487f64f9f126e4696b2e15321e290e2ffbf07...,7b4a2ca5b49793285cc1c31d6abbff691a2f2295ef12f4...
5,6,task_geometry,ed196c011bd0c1c04f21adbda2b7d9ad359e26a3fd036e...,f62a5913631aef3de013f989a62c78611c0f9ff28bed09...
6,7,training_curves,e06fafcdfaca0359f7e6d9126976727fe72f55794dda37...,3c2d86844569587964e2b0d8107166071be1f59daea357...
7,8,seed_stability,14da7dec903566ba4f9cc7126fc86a4fb70302f66e5e05...,a31f88b9cee7d522b21e9e0d36a296e1137cf7bfc4ee6d...
8,9,backbone_leaderboard,b802e4dc8b44fc9d5c0628cd838f5754fe2d1a80031a8c...,6bb70902f1d9e84e9bef8965bbe042d979e1681f74632f...
9,10,backbone_heatmap,976184eeb225e2864796570080556416ea0404c2bf92e8...,0c71fbd5f8d15df20d2b14e6c9c87742636a1981d5078b...


## Interpretation boundary

The machine-generated decision applies the preregistered null, 1% SESOI, dataset-win, robustness, and integrity conditions. Secondary slices do not alter that decision.